<div style="text-align: center;" >
<h1 style="margin-top: 0.2em; margin-bottom: 0.1em;">Introduction to Computation for the Social Sciences</h1>
<h2 style="margin-top: 0.7em; margin-bottom: 0.3em;">Assignment 3</h2>
<h3 style="margin-top: 0.7em; margin-bottom: 0.3em;">Deadline: 22.12.2024 12:00 pm</h3>

</div>
<br>

<h4 style="margin-top: 0.7em; margin-bottom: 0.3em; font-style:italic">
Please push your solutions to your personal repository in our <a href='https://classroom.github.com/a/tGD_7t85'>GitHub Classroom</a></h4><br>

***

This assignment will test your knowledge in object oriented programming (OOP), your understanding of regular expressions (regexes), and finally we will have you perform sentiment analysis.<br>
As always: In case of questions feel free to reach out to us tutors in person, via mail, or over discord.<br>
***Important: Submit a solution for every single task and do not skip any of them. Even if your solution is not perfect or doesn't work you might still receive some points that way!***


***

In [1]:
# Import the modules you use throughout the assignment here (this is called a setup chunk/cell)
import pandas as pd
import numpy as np
import requests
import json
import re
import tkinter as tk
import random
from tkinter import messagebox
import nltk
# nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
from wordcloud import WordCloud


# Part 1 - Object Oriented Programming

## Task 1.1


The [Dog API](https://dogapi.dog/docs/api-v2) is a simple and easy to use tool to get information about certain dog breeds along with random facts about dogs.<br>
Use the API to create a class `DogBreed`.
- This class takes as input the name of a dog breed (e.g. 'Caucasian Shepherd Dog' or 'Bouvier des Flandres')
- Implement a class method `get_breed_info()`that returns a description of input dog breed
- Implement a class method `get_max_age()`that returns the maximum life expectancy of the input dog breed
- Implement a class method `get_fact()`that returns a random fact about dogs

In [ ]:
# STUDENT code
# Your code

# Create the class DogBreed
class DogBreed:
    """
    A class about a dog breed.

    Attributes
    ----------
    name: str
        The name of a dog breed.


    Methods
    -------
    __init__(name):
        Initializes the class with the name of a dog breed.
    get_breed_info():
        Filters for the information about a certain dog breed and returns a description of the dog breed.
    get_max_age():
        Filters for the information about a certain dog breed and returns the maximum life expectancy of the dog breed.
    get_fact():
        Retrieves a random fact about dogs via the Dog API and returns the fact.
    @staticmethod get_all_breeds(url='https://dogapi.dog/api/v2/breeds?page[number]=1', first_call=True, breeds=None):
        Retrieves information about numerous dog breeds via the Dog API and returns the information about the dog breeds.
    
    """

    # Initialize the class DogBreed
    def __init__(self, name):
        # Store the name of the input dog breed as an attribute of the class object
        self.name = name

    # Define a method to get a description about the input dog breed
    def get_breed_info(self):
        # Store the return value of the static method get_all_breeds() 
        # as an object (= dictionary) named breeds
        breeds = self.get_all_breeds()
        # In the object (= dictionary) breeds, search for the input dog breed and 
        # store the information about the input dog breed as an object named breed
        breed = [breed for breed in breeds if breed['attributes']['name'] == self.name]
        # Return the description about the input dog breed
        # (because breed is a list of dictionaries, I had to index it with [0] and 
        # then use the keys 'attributes' and 'description' to retrieve the description)
        return breed[0]['attributes']['description']

    # Define a method to get the maximum life expectancy of the input dog breed
    def get_max_age(self):
        # Store the return value of the static method get_all_breeds() 
        # as an object (= dictionary) named breeds
        breeds = self.get_all_breeds()
        # In the object (= dictionary) breeds, search for the input dog breed and 
        # store the information about the input dog breed as an object named breed
        breed = [breed for breed in breeds if breed['attributes']['name'] == self.name]
        # Return the maximum life expectancy of the input dog breed
        # (because breed is a list of dictionaries, I had to index it with [0] and 
        # then use the keys 'attributes', 'life' and 'max' to retrieve the maximum life expectancy)
        return breed[0]['attributes']['life']['max']

    # Define a method to get a random fact about dogs
    def get_fact(self):
        # Send a request for a dog fact to the Dog API and store the response in an object named resp
        resp = requests.get('https://dogapi.dog/api/v2/facts')
        # Store the content of the response in an object named fact
        fact = resp.json() 
        # Return the random fact about dogs
        # (because of the data structure of the object fact, I had to use the key 'data', 
        # index the value of the key with [0] and then use the keys 'attributes' and 'body' 
        # to retrieve the random fact)
        return fact['data'][0]['attributes']['body']

    # Define a static method to get information about all available dog breeds via the Dog API
    # (To be able to send a request for a certain dog breed to the Dog API, I would need to know
    # the id of the dog breed. Since I do not know the ids of dog breeds, I instead sent a request
    # for the information about all available dog breeds to the Dog API. This allows me to filter for
    # the information about the input dog breed afterwards.)
    @staticmethod
    def get_all_breeds(url='https://dogapi.dog/api/v2/breeds?page[number]=1', first_call=True, breeds=None):
        # Send a request to the Dog API for the information about dog breeds
        # (There are 29 pages of information about dog breeeds. This request is only for one of those pages.)
        resp = requests.get(url)
        # Store the content of the response in an object named page
        page = resp.json()  

        # The method get_all_breeds works with recursion. If this is the first call of the method
        # (the default input for first_call is True):
        if first_call:
            # Store the information about dog breeds ('data') of the first page in an 
            # object (= list) named breeds
            breeds = page['data']
        # If this is not the first call of the function:
        else:
            # Add the information about dog breeds ('data') of this page to the list breeds
            breeds = breeds + page['data']

        # If this is not the last page of the Dog API 
        # (in other words: If there is a link to the last page; this is true for all pages except
        # the last page)
        if 'last' in page['links']:
            # Store the link to the next page of the Dog API in a variable named next_page
            next_page = page['links']['next']
            # Call the method get_all_breeds again; this time with the url of the next page, 
            # set first_call to false (because then this is not the first call of the method)
            # and input the breeds that have already been retrieved
            return DogBreed.get_all_breeds(url=next_page, first_call=False, breeds=breeds)

        # After all 29 pages of the Dog API have been processed, return the final object breed
        # with the information about all avaiable dog breeds in the Dog API
        return breeds



In [ ]:
# Create an object named dog of the class DogBreed, using the dog breed 'Soft Coated Wheaten Terrier'
dog = DogBreed('Soft Coated Wheaten Terrier')

In [ ]:
# Call the class method get_breed_info() for the dog
dog.get_breed_info()

In [ ]:
# Call the class method get_max_age() for the dog
dog.get_max_age()

In [ ]:
# Call the class method get_fact() for the dog (even though the random fact is not specifically 
# about that dog breed)
dog.get_fact()

## Task 1.2 - Bonus


The class `DogBreed` you created in task 1.1 is probably pretty simple. To remedy this, please implement a system that catches exceptions and deals with them appropriately. For this you will have to think about the types of exceptions someone using the `DogBreed` class might encounter and how you want to deal with each of them.<br>
Feel free to do additional on how to handle exceptions as we did not cover them in much detail. You can get a nice overview, for example, [here](https://www.w3schools.com/python/python_try_except.asp).<br>
Please copy paste your code from above and implement exception handling below. Of course you can also just rewrite the class ;)

***Tip:***
Error handling is usually done something like this in python:<br>
```
try:
    some_code
    return some_var
except error1 as e:
    handle specific exception
except error2 as e:
    handle specific exception
except:
    handle general exception
```

Before you start to code, as mentioned above, you will have to think about the different exceptions someone might encounter and how you would deal with them.
E.g.:<br>
Person tries to use method that is not implemented -> print('This method does not exist for class DogBreed')...<br>

In [ ]:
# STUDENT CODE (since it is better than my solution)
# Comment: Check whether overwriting of build-in error messages is possible or adjust
# the custom error message to reflect the fact that there is already a build-in error message


# Note: To make it clear what I have changed, I comment on the changes with capital letters.
    
class DogBreed:
    """
    A class about a dog breed.

    Attributes
    ----------
    name: str
        The name of a dog breed.


    Methods
    -------
    __init__(name):
        Initializes the class with the name of a dog breed.
    get_breed_info():
        Filters for the information about a certain dog breed and returns a description of the dog breed.
    get_max_age():
        Filters for the information about a certain dog breed and returns the maximum life expectancy of the dog breed.
    get_fact():
        Retrieves a random fact about dogs via the Dog API and returns the fact.
    @staticmethod get_all_breeds(url='https://dogapi.dog/api/v2/breeds?page[number]=1', first_call=True, breeds=None):
        Retrieves information about numerous dog breeds via the Dog API and returns the information about the dog breeds.
    
    """

    # Initialize the class DogBreed
    def __init__(self, name=None):
        # IF NO DOG BREED IS PROVIDED:
        if name is None:
            # PRINT 'Please provide a dog breed.'
            return print('Please provide a dog breed.')  
        # Store the name of the input dog breed as an attribute of the class object
        self.name = name

    # Define a method to get a description about the input dog breed
    def get_breed_info(self):
        # Store the return value of the static method get_all_breeds() 
        # as an object (= dictionary) named breeds
        breeds = self.get_all_breeds()
        # In the object (= dictionary) breeds, search for the input dog breed and 
        # store the information about the input dog breed as an object named breed
        breed = [breed for breed in breeds if breed['attributes']['name'] == self.name]
        # TRY:
        try: 
            # return the description about the input dog breed
            # (because breed is a list of dictionaries, I had to index it with [0] and 
            # then use the keys 'attributes' and 'description' to retrieve the description)
            return breed[0]['attributes']['description']
        # IF THE LIST breed IS EMPTY, THERE IS AN INDEX ERROR WHEN TRYING TO ACCESS THE FIRST LIST ITEM;
        # STORE THE INDEX ERROR AS e
        except IndexError as e: 
            # PRINT THE ERROR AND TELL THE USER THAT THERE IS NO INFORMATION ABOUT THIS DOG BREED
            print(f"Got index error: {e}\nUnfortunately, there is no information about this dog breed.")


    # Define a method to get the maximum life expectancy of the input dog breed
    def get_max_age(self):
        # Store the return value of the static method get_all_breeds() 
        # as an object (= dictionary) named breeds
        breeds = self.get_all_breeds()
        # In the object (= dictionary) breeds, search for the input dog breed and 
        # store the information about the input dog breed as an object named breed
        breed = [breed for breed in breeds if breed['attributes']['name'] == self.name]
        # TRY:
        try: 
            # return the maximum life expectancy of the input dog breed
            # (because breed is a list of dictionaries, I had to index it with [0] and 
            # then use the keys 'attributes', 'life' and 'max' to retrieve the maximum life expectancy)
            return breed[0]['attributes']['life']['max']
        # IF THE LIST breed IS EMPTY, THERE IS AN INDEX ERROR WHEN TRYING TO ACCESS THE FIRST LIST ITEM;
        # STORE THE INDEX ERROR AS E
        except IndexError as e: 
            # PRINT THE ERROR AND TELL THE USER THAT THERE IS NO INFORMATION ABOUT THIS DOG BREED
            print(f"Got index error: {e}\nUnfortunately, there is no information about this dog breed.")

    # Define a method to get a random fact about dogs
    def get_fact(self):
        # Send a request for a dog fact to the Dog API and store the response in an object named resp
        resp = requests.get('https://dogapi.dog/api/v2/facts')
        # IF THE REQUEST WAS NOT SUCCESSFUL (I.E. STATUS CODE IS NOT 200)
        if resp.status_code != 200:
            # INFORM THE USER ABOUT THE UNSUCCESSFUL REQUEST AND PRINT THE STATUS CODE
            return print(f"Request to the Dog API was not successful: Status code {resp.status_code}")
        # Store the content of the response in an object named fact
        fact = resp.json() 
        # return the random fact about dogs
        # (because of the data structure of the objet fact, I had to use the key 'data', 
        # index the value of the key with [0] and then use the keys 'attributes' and 'body' 
        # to retrieve the random fact)
        return fact['data'][0]['attributes']['body']

    # Define a static method to get information about all available dog breeds via the Dog API
    # (To be able to send a request for a certain dog breed to the Dog API, I would need to know
    # the id of the dog breed. Since I do not know the ids of dog breeds, I instead sent a request
    # for the information about all available dog breeds to the Dog API. This allows me to filter for
    # the information about the input dog breed afterwards.)
    @staticmethod
    def get_all_breeds(url='https://dogapi.dog/api/v2/breeds?page[number]=1', first_call=True, breeds=None):
        # Send a request to the Dog API for the information about dog breeds
        # (There are 29 pages of information about dog breeeds. This request is only for one of those pages.)
        resp = requests.get(url)
        # IF THE REQUEST WAS NOT SUCCESSFUL (I.E. STATUS CODE IS NOT 200)
        if resp.status_code != 200:
            # INFORM THE USER ABOUT THE UNSUCCESSFUL REQUEST AND PRINT THE STATUS CODE
            return print(f"Request to the Dog API was not successful: Status code {resp.status_code}")
        # Store the content of the response in an object named page
        page = resp.json()  

        # The method get_all_breeds works with recursion. If this is the first call of the method
        # (the default input for first_call is True):
        if first_call:
            # Store the information about dog breeds ('data') of the first page in an 
            # object (= list) named breeds
            breeds = page['data']
        # If this is not the first call of the function:
        else:
            # Add the information about dog breeds ('data') of this page to the list breeds
            breeds = breeds + page['data']

        # If this is not the last page of the Dog API 
        # (in other words: If there is a link to the last page; this is true for all pages except
        # the last page)
        if 'last' in page['links']:
            # Store the link to the next page of the Dog API in a variable named next_page
            next_page = page['links']['next']
            # Call the method get_all_breeds again; this time with the url of the next page, 
            # set first_call to false (because then this is not the first call of the method)
            # and input the breeds that have already been retrieved
            return DogBreed.get_all_breeds(url=next_page, first_call=False, breeds=breeds)

        # After all 29 pages of the Dog API have been processed, return the final object breed
        # with the information about all avaiable dog breeds in the Dog API
        return breeds

    # MAKE SURE THAT A METHOD OR ATTRIBUTE THAT IS CALLED REALLY EXISTS
    def __getattr__(self, method_or_attribute):
        # TRY
        try:
            # IF THE METHOD OR ATTRIBUTE EXISTS, THE METHOD IS EXECUTED OR THE ATTRIBUTE IS RETURNED
            return super().__getattribute__(method_or_attribute)
        # IF THE METHOD DOES NOT EXIST, AN ATTRIBUTE ERROR IS RAISED
        except AttributeError as e:
            # PRINT THE ATTRIBBUTE ERROR AND TELL THE USER THAT THE METHOD OR ATTRIBUTE DOES NOT EXIST
            print(f"Got attribute error: {e}\nUnfortunately, this method or attribute does not exist for class DogBreed.")

In [ ]:
# If no dog breed is provided:
dog2 = DogBreed()

In [ ]:
# If the dog breed is not available in the Dog API:
dog3 = DogBreed('Sheepadoodle')

dog3.get_breed_info()

In [ ]:
# If the dog breed is not available in the Dog API:
dog4 = DogBreed('Sheepadoodle')

dog4.get_max_age()

In [ ]:
# If the method or attribute (in this case: attribute) does not exist in the class DogBreed

# This dog breed does exist (see above)
dog5 = DogBreed('Soft Coated Wheaten Terrier')

# But this attribute does not exist
dog5.fluffy

In [ ]:
# Just to make sure: The class methods and attributes still work properly after my changes
print(dog.get_breed_info(), "\n")
print(dog.get_max_age(), "\n")
print(dog.get_fact(), "\n")
print(dog.name)

***
# Part 2 - Preprocessing & Regular Expressions

In the assignemnt folder, you will find a cdc_whistleblower_tweets.zip containing a .json file with tweets from the CDC Whistleblower debate. The file consists of 272660 tweets containing the hashtag #CDCWhistleblower. In case you're interested in the data you can get some additional infos [here](https://www.snopes.com/fact-check/bad-medicine/).<br>

You will need to process this file aka. the tweets in it in order to work with the data in task 3 & task 4.<br>
***Hint: Use a sample of tweets in order to test your preprocessing steps before you run them on the whole data set (eg. 1000 tweets)***

## Task 2.1

In task 4 you will have to perform sentiment analysis on these tweets. Keeping that in mind please perform the following preprocessing operations on the data. <br>
First, we will replace user mentions, extract hashtags and remove urls from the original text column 
To do so, you will need to use regular expressions.

Create three functions:
- `replace_mentions()`: a function that replaces all user mentions (@username) in the text column with '@user' (think of it as anonymizing the data). You are supposed to change the original text column here!
- `extract_hastags()`: a function that extracts all hashtags (e.g. #CDCWhistleblower) appearing in the tweet and stores them in a new column `hashtags` without altering the original text column
- `remove_urls()`: a function that extracts all domain names from the 'expanded_url' field (entities → urls → expanded_url) to a new column (e.g. [google.com, wikipedia.org, ...]) and removes the URLs from the original text colmn.<br>



Before you get into coding take a moment and think about how you might use regular expressions to solve this task. 
Develop a regular expression for each of the three functions and shortly explain how it works - for example in the comments.




For this task, you might want to consider the [re module](https://docs.python.org/3/library/re.html).<br>


In [5]:
# Load the twitter data
with open('cdc_whistleblower_tweets.json', 'r') as openfile:
 
    # Reading from json file
    cdc_data = json.load(openfile)

# check how the data looks
cdc_data

In [ ]:
# Turn dictionary into a Dataframe
cdc_data = pd.DataFrame(cdc_data)
type(cdc_data)

In [ ]:
# LET'S rather use a student solution (since it is clearly better) - AJ

## Task 2.2

As a second step of preprocessing, we now want to normalize and tokenize the text as well as to remove stop words. 
These preprocessings should NOT change the original `text` column. Instead, write the functions and apply them in way that the result is stored in a new column `clean_tokens`. 


- Write a `normalize_text()` function that transforms all text to lower case
- Write a `tokenize_text()` function that tokenizes the text aka. splits it into individual words
- Write a `remove_stopwords()` function that removes all stopwords that are part of `eng_stop_words.txt`


At the end, make sure you save the preprocessed data in a file so as not to have to run the preprocessing everytime you come back to the assignment. <br>
Run the preprocessing on the whole data set. Save the preprocessed data as 'processed_tweets.json'.<br>

This file will exceed the 100MB GitHub upload limit. You will have to create a compressed .zip directory containing the file in order to upload it. Before uploading you will then have to delete or move the uncompressed file from your personal repository. If you do not do this you won't be able to upload your submission.<br>
Another solution would be to open the file '.gitignore' which we put in your personal repository at the start of the semester. There you will see '.ipynb_checkpoints/'. Copy the name of your .json file and add it as a new line to this '.gitignore' file (e.g. cdc_whistleblower_tweets.json). This will tell git to ignore the file and enable you to upload your submission.

In [3]:
# Your code


## Task 2.3 - Bonus



Building on your preprocessing in this task, print the data set's 10 most frequently occurring words/tokens, hashtags, and domain names.<br>
Use the [wordcloud module](https://python-course.eu/applications-python/python-wordcloud-tutorial.php) to visualize your findings regarding words/tokens and hashtags. Create one plot for words/tokens and one for hashtags.

In [ ]:
# Your code


***
# Part 3 - Application

Using the [Tkinter](https://docs.python.org/3/library/tkinter.html) library write the code for an application. This application is supposed to aid you in labelling texts according to their sentiment. For this you will need to implement certain functions into your app. Once your application is up an running sample 150 tweets from the CDCWhistleblower data set and label them using your app.<br>
Save your sample as .json file and be sure to include the tweet_id for each tweet so you are able to compare your labels with the ones you get from running VADER in task 4.<br>
Please use the original text for your labeling. After the preprocessing in task 2 it should not contain specific user mentions but only '@user', there should be no URLs left in the text, and #Hashtags should appear untouched in your texts.<br>

***Hint: Before you actually start coding, a great approach is to first of all start by conceptualizing your small app. Think about which functionalities you need to have and how these depend on and call each other. You can, as an example, refer to the extended Tkinter example we covered in the tutorial.***
If you like to, you can hand in your handwritten schema.

![Picture_of_schema](path_to_the_pic/pic.jpg) <-- **double click this cell to see how appending pictures with .ipynb files works. Make sure to hand in the png as well.**

## Task 3.1


In order for your application to work as intended follow the steps below. Note that this is no easy task! Work with other students, exchange ideas, or find help online but remember to only hand in code that you have written yourself!<br>

Implement the application as a class:<br>
- that takes 2 arguments: 'input_file' and 'output_file'
- that supports the method `load_data()` which loads the output_file and the input_file and removes tweets that are already classified aka. part of the output file. This is to ensure that you won't label the same text twice and that you will reach an end, once every text got labeled once. 
- with method `start_app()` that runs your application with a text box showing the content (text) of a tweet and 4 labeling buttons (Positive, Negative, Neutral, Undecidable) used to classify the content of each tweet
- with method `next_text()` that chooses a random tweet from the input data, displays it in the text box of the app and once the tweet has been classified, removes it from the pool of tweets still to be labeled<br>

Feel free to implement any other methods you deem necessary and do not forget to comment and explain your code!<br>

***Hint: There are many ways of solving this task! There is no 'one and only' solution so be bold and experiment with your ideas. You can probably make most of them work!***

In [6]:
# Your code


## Task 3.2


Use your application to classify your sample of 150 tweets from the CDCWhisleblower data and save your result as 'manually_labeled_tweets.json'.<br>
Make sure you upload this labeled sample of tweets to your solutions repository!

***
# Part 4 - Sentiment Analysis

In this last part of the assignment you will use the VADER sentiment analyzer to, well, analyze the sentiment of the CDCWhistleblower tweets.<br>
You can find a tutorial of how the nltk module's VADER analyzer works [here](https://www.nltk.org/howto/sentiment.html).<br>

## Task 4.1


Use the sentiment analyzer on the preprocessed original text column (the one without URLs, with @user, and #Hashtags) and save the resulting polarity score to a new column.

In [8]:
# Your code


## Task 4.2


Create another column containing the VADER sentiment label for each tweet. You can get the label (positive, negative, neutral) by checking the polarity scores you got in task 4.1. If the polarity score is smaller 0 (<0) the label should be 'negative', if it is larger 0 (>0) the label should be 'positive', and otherwise the label should be 'neutral'.<br>
What are the shares of 'positive', 'negative', and 'neutral' tweets? Visualize your findings.

In [9]:
# Your code


## Task 4.3



Match the 150 tweets you labeled manually with the labels you got from applying the VADER sentiment analyzer to the data. Visualize how VADER performed at analyzing the sentiment against your manual classifications of these 150 tweets.

In [10]:
# Your code
